# Análise de Folha de Pagamento de RH | HR Payroll Analysis
**EN:** Advanced SQL (CTEs + window functions) on 100% synthetic data, using DuckDB, Pandas and Plotly.
**PT:** SQL avançado (CTEs + window functions) em dados 100% sintéticos, com DuckDB, Pandas e Plotly.

**Reproducible / Reprodutível:** Runtime > Run all (Executar tudo). Fixed seed = same results every time / seed fixa = mesmos resultados sempre.
**Security / Segurança:** no keys, no real personal data, no external connections / sem chaves, sem dados pessoais reais, sem conexões externas.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klaytonsi/analise-folha-pagamento-rh/blob/main/analise_folha_pagamento.ipynb)

In [ ]:
# EN: Install lightweight libraries (pandas/numpy already come with Colab)
# PT: Instala bibliotecas leves (pandas/numpy já vêm no Colab)
!pip install -q duckdb plotly

In [ ]:
# EN: Imports and fixed seed for reproducibility
# PT: Imports e seed fixa para reprodutibilidade
import os
import numpy as np
import pandas as pd
import duckdb
import plotly.express as px

SEED = 42                      # EN: fixed seed | PT: seed fixa
N_EMPLOYEES = 1500             # EN: small size = light | PT: tamanho pequeno = leve
rng = np.random.default_rng(SEED)

## 1. Synthetic data / Dados sintéticos
EN: Employees are only numeric IDs (no names, CPF or e-mails). PT: Funcionários são apenas IDs numéricos (sem nomes, CPF ou e-mails).

In [ ]:
# EN: Departments table | PT: Tabela de departamentos
departments = pd.DataFrame({
    "department_id": [1, 2, 3, 4, 5],
    "department": ["TI", "Financeiro", "RH", "Comercial", "Operações"],
})

# EN: Job titles with base salary (BRL) | PT: Cargos com salário base (R$)
jobs = pd.DataFrame({
    "job_id": range(1, 11),
    "job_title": ["Analista TI", "Coordenador TI", "Assistente Fin.", "Analista Fin.",
                  "Assistente RH", "Analista RH", "Vendedor", "Gerente Comercial",
                  "Operador", "Supervisor Operações"],
    "department_id": [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
    "base_salary": [4500, 9000, 3800, 7500, 3500, 6800, 3200, 8000, 2600, 5200],
})

# EN: Probability of each job (sums to 1) | PT: Probabilidade de cada cargo (soma 1)
job_weights = [0.08, 0.04, 0.07, 0.04, 0.04, 0.02, 0.15, 0.06, 0.35, 0.15]

# EN: Build employees: job, salary with noise, hire and termination dates
# PT: Monta funcionários: cargo, salário com variação, datas de admissão e desligamento
employees = pd.DataFrame({"employee_id": np.arange(1, N_EMPLOYEES + 1)})
employees["job_id"] = rng.choice(jobs["job_id"].to_numpy(), size=N_EMPLOYEES, p=job_weights)
employees = employees.merge(jobs[["job_id", "department_id", "base_salary"]], on="job_id", how="left")
employees["salary"] = (employees["base_salary"] * rng.lognormal(0, 0.12, N_EMPLOYEES)).round(2)
employees = employees.drop(columns="base_salary")

start, end = pd.Timestamp("2020-01-01"), pd.Timestamp("2025-12-31")
span_days = (end - start).days
employees["hire_date"] = start + pd.to_timedelta(rng.integers(0, span_days + 1, N_EMPLOYEES), unit="D")

# EN: ~28% leave; those who would leave after the end date stay active
# PT: ~28% saem; quem sairia depois da data final continua ativo
leaves = rng.random(N_EMPLOYEES) < 0.28
term = employees["hire_date"] + pd.to_timedelta(rng.integers(60, 900, N_EMPLOYEES), unit="D")
employees["termination_date"] = term.where(leaves & (term <= end), pd.NaT)

print(employees.shape, "| ativos / active:", employees["termination_date"].isna().sum())
employees.head()

(1500, 6) | ativos / active: 1140


,employee_id,job_id,department_id,salary,hire_date,termination_date
0,1,9,5,2918.49,2020-01-11,NaT
1,2,7,4,2933.50,2023-08-13,NaT
2,3,10,5,4693.64,2023-11-05,2025-04-24
3,4,9,5,2747.46,2022-04-08,NaT
4,5,2,1,10133.54,2022-10-25,NaT


## 2. Load into DuckDB / Carregar no DuckDB

In [ ]:
# EN: In-memory database, nothing is saved to disk | PT: Banco em memória, nada é salvo em disco
con = duckdb.connect(":memory:")
con.register("departments", departments)
con.register("jobs", jobs)
con.register("employees", employees)

# EN: View = one row per employee per active month (2023-2025)
# PT: View = uma linha por funcionário por mês ativo (2023-2025)
con.execute("""
CREATE OR REPLACE VIEW payroll AS
SELECT CAST(r.month_ts AS DATE) AS month_start,
       e.employee_id, e.department_id, e.salary
FROM range(DATE '2023-01-01', DATE '2026-01-01', INTERVAL 1 MONTH) AS r(month_ts)
JOIN employees e
  ON e.hire_date <= last_day(CAST(r.month_ts AS DATE))
 AND (e.termination_date IS NULL OR e.termination_date >= CAST(r.month_ts AS DATE))
""")
con.execute("SELECT COUNT(*) AS rows_payroll FROM payroll").df()

,rows_payroll
0,32562


## 3. Payroll cost by department and month / Custo de folha por departamento e mês

In [ ]:
# EN: CTE + LAG (month-over-month) + running total per year
# PT: CTE + LAG (variação mês a mês) + acumulado no ano
q1 = con.execute("""
WITH monthly AS (
    SELECT p.month_start, d.department,
           COUNT(*) AS headcount,
           ROUND(SUM(p.salary), 2) AS payroll_cost
    FROM payroll p
    JOIN departments d ON d.department_id = p.department_id
    GROUP BY 1, 2
)
SELECT month_start, department, headcount, payroll_cost,
       ROUND(payroll_cost - LAG(payroll_cost) OVER (PARTITION BY department ORDER BY month_start), 2) AS mom_change,
       ROUND(SUM(payroll_cost) OVER (PARTITION BY department, YEAR(month_start) ORDER BY month_start), 2) AS ytd_cost
FROM monthly
ORDER BY department, month_start
""").df()
q1.tail(6)

,month_start,department,headcount,payroll_cost,mom_change,ytd_cost
174,2025-07-01,TI,135,865960.75,13916.21,5944547.19
175,2025-08-01,TI,140,887957.37,21996.62,6832504.56
176,2025-09-01,TI,140,882166.42,-5790.95,7714670.98
177,2025-10-01,TI,141,882669.82,503.40,8597340.80
178,2025-11-01,TI,143,897268.79,14598.97,9494609.59
179,2025-12-01,TI,143,902733.84,5465.05,10397343.43


In [ ]:
# EN: Chart: payroll cost over time | PT: Gráfico: custo de folha ao longo do tempo
fig1 = px.line(q1, x="month_start", y="payroll_cost", color="department",
               title="Custo mensal de folha por departamento | Monthly payroll cost by department",
               labels={"month_start": "Mês | Month", "payroll_cost": "R$"})
fig1.show()

## 4. Top 3 salaries per department / Top 3 salários por departamento

In [ ]:
# EN: DENSE_RANK inside each department (active employees only)
# PT: DENSE_RANK dentro de cada departamento (somente ativos)
q2 = con.execute("""
WITH active AS (
    SELECT e.employee_id, d.department, j.job_title, e.salary
    FROM employees e
    JOIN departments d ON d.department_id = e.department_id
    JOIN jobs j ON j.job_id = e.job_id
    WHERE e.termination_date IS NULL
),
ranked AS (
    SELECT *, DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_rank
    FROM active
)
SELECT * FROM ranked WHERE salary_rank <= 3 ORDER BY department, salary_rank
""").df()
q2

,employee_id,department,job_title,salary,salary_rank
0,729,Comercial,Gerente Comercial,10183.34,1
1,754,Comercial,Gerente Comercial,9903.02,2
2,1321,Comercial,Gerente Comercial,9582.92,3
3,1367,Financeiro,Analista Fin.,9355.47,1
4,970,Financeiro,Analista Fin.,9332.94,2
5,1457,Financeiro,Analista Fin.,9061.31,3
6,986,Operações,Supervisor Operações,7107.10,1
7,905,Operações,Supervisor Operações,7046.19,2
8,1279,Operações,Supervisor Operações,6984.18,3
9,157,RH,Analista RH,9646.86,1


## 5. Monthly turnover / Turnover mensal

In [ ]:
# EN: Turnover % = terminations / headcount, plus a 3-month moving average
# PT: Turnover % = desligamentos / headcount, mais média móvel de 3 meses
q3 = con.execute("""
WITH hc AS (
    SELECT month_start, COUNT(*) AS headcount FROM payroll GROUP BY 1
),
terms AS (
    SELECT CAST(DATE_TRUNC('month', termination_date) AS DATE) AS month_start, COUNT(*) AS terminations
    FROM employees WHERE termination_date IS NOT NULL GROUP BY 1
),
base AS (
    SELECT hc.month_start, hc.headcount,
           COALESCE(t.terminations, 0) AS terminations,
           ROUND(100.0 * COALESCE(t.terminations, 0) / hc.headcount, 2) AS turnover_pct
    FROM hc LEFT JOIN terms t USING (month_start)
)
SELECT *, ROUND(AVG(turnover_pct) OVER (ORDER BY month_start ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS turnover_ma3
FROM base ORDER BY month_start
""").df()

fig2 = px.bar(q3, x="month_start", y="turnover_pct",
              title="Turnover mensal (%) | Monthly turnover (%)",
              labels={"month_start": "Mês | Month", "turnover_pct": "%"})
fig2.add_scatter(x=q3["month_start"], y=q3["turnover_ma3"], mode="lines", name="Média móvel 3m | 3m moving avg")
fig2.show()

## 6. Above job average / Acima da média do cargo

In [ ]:
# EN: AVG() OVER (PARTITION BY job) to flag salaries 15%+ above the job average
# PT: AVG() OVER (PARTITION BY cargo) para marcar salários 15%+ acima da média do cargo
q4 = con.execute("""
WITH active AS (
    SELECT e.employee_id, j.job_title, e.salary
    FROM employees e JOIN jobs j ON j.job_id = e.job_id
    WHERE e.termination_date IS NULL
),
stats AS (
    SELECT *, AVG(salary) OVER (PARTITION BY job_title) AS job_avg FROM active
)
SELECT employee_id, job_title, salary,
       ROUND(job_avg, 2) AS job_avg,
       ROUND(100.0 * (salary / job_avg - 1), 1) AS pct_above_avg
FROM stats
WHERE salary > job_avg * 1.15
ORDER BY pct_above_avg DESC
LIMIT 15
""").df()
q4

,employee_id,job_title,salary,job_avg,pct_above_avg
0,1404,Vendedor,4619.54,3215.51,43.7
1,157,Analista RH,9646.86,6873.23,40.4
2,707,Operador,3643.81,2597.48,40.3
3,683,Analista TI,6224.89,4481.23,38.9
4,877,Vendedor,4460.42,3215.51,38.7
5,310,Assistente Fin.,5323.60,3854.15,38.1
6,986,Supervisor Operações,7107.10,5162.10,37.7
7,1316,Operador,3566.84,2597.48,37.3
8,905,Supervisor Operações,7046.19,5162.10,36.5
9,837,Operador,3540.31,2597.48,36.3


## 7. Export for Looker Studio / Exportar para o Looker Studio

In [ ]:
# EN: One tidy table: month x department with headcount, cost, hires, terminations
# PT: Uma tabela limpa: mês x departamento com headcount, custo, admissões, desligamentos
summary = con.execute("""
WITH base AS (
    SELECT month_start, department_id, COUNT(*) AS headcount, SUM(salary) AS payroll_cost
    FROM payroll GROUP BY 1, 2
),
t AS (
    SELECT CAST(DATE_TRUNC('month', termination_date) AS DATE) AS month_start, department_id, COUNT(*) AS terminations
    FROM employees WHERE termination_date IS NOT NULL GROUP BY 1, 2
),
h AS (
    SELECT CAST(DATE_TRUNC('month', hire_date) AS DATE) AS month_start, department_id, COUNT(*) AS hires
    FROM employees GROUP BY 1, 2
)
SELECT b.month_start, d.department, b.headcount,
       ROUND(b.payroll_cost, 2) AS payroll_cost,
       COALESCE(h.hires, 0) AS hires,
       COALESCE(t.terminations, 0) AS terminations
FROM base b
JOIN departments d ON d.department_id = b.department_id
LEFT JOIN t ON t.month_start = b.month_start AND t.department_id = b.department_id
LEFT JOIN h ON h.month_start = b.month_start AND h.department_id = b.department_id
ORDER BY b.month_start, d.department
""").df()

os.makedirs("output", exist_ok=True)
summary.to_csv("output/resumo_dashboard.csv", index=False)
print(summary.shape)
summary.head()

(180, 6)


,month_start,department,headcount,payroll_cost,hires,terminations
0,2023-01-01,Comercial,141,651381.72,3,2
1,2023-01-01,Financeiro,65,345862.55,3,0
2,2023-01-01,Operações,324,1124641.22,14,4
3,2023-01-01,RH,37,178570.56,0,0
4,2023-01-01,TI,80,518832.12,1,1


In [ ]:
# EN: Download the CSV (works only in Colab) | PT: Baixa o CSV (funciona só no Colab)
try:
    from google.colab import files
    files.download("output/resumo_dashboard.csv")
except ImportError:
    print("Not in Colab: file is in output/ | Fora do Colab: arquivo está em output/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Business conclusions / Conclusões de negócio

**1. Cost grows with headcount, not with pay / Custo cresce com o headcount, não com salário.**
PT: a folha mensal subiu de R$ 2,82 mi (jan/23) para R$ 4,85 mi (dez/25), +72%; o custo anual foi de R$ 37,8 mi (2023) para R$ 46,5 mi (2024, +23%) e R$ 54,8 mi (2025, +18%). O headcount passou de 647 para 1.144 (+77%) e o custo médio por pessoa ficou estável (R$ 4.357 → R$ 4.238), então a alta vem das contratações, não de reajustes (os dados sintéticos não têm aumentos salariais).
EN: monthly payroll rose from BRL 2.82M (Jan/23) to BRL 4.85M (Dec/25), +72%; headcount went from 647 to 1,144 (+77%) and cost per head stayed flat, so growth is driven by hiring, not raises (the synthetic data has no pay increases).

**2. Operations is the biggest bill; TI is the most expensive per head / Operações pesa mais; TI é o mais caro por pessoa.**
PT: Operações responde por 39,2% da folha de dez/25 (567 pessoas, R$ 3.352 por pessoa); TI tem 18,6% da folha com apenas 143 pessoas (R$ 6.313 por pessoa). Em crescimento de custo desde jan/23, Financeiro lidera (+82,1%) e RH cresce menos (+47,3%).
EN: Operations is 39.2% of Dec/25 payroll (567 people, BRL 3,352 each); TI is 18.6% with only 143 people (BRL 6,313 each). Finance grew fastest since Jan/23 (+82.1%), HR slowest (+47.3%).

**3. Turnover is low and stable, with two spikes / Turnover baixo e estável, com dois picos.**
PT: turnover anual (desligamentos / headcount médio) de 7,5% em 2023, 9,1% em 2024 e 6,8% em 2025; média mensal de 0,66%. Picos em mar/23 (1,47%) e jan/24 (1,44%). Por departamento (2023-2025, anualizado), Financeiro (8,6%) e Operações (8,3%) ficam acima de TI (6,6%) e RH (7,0%), mas a diferença é pequena.
EN: annual turnover (terminations / average headcount) was 7.5% in 2023, 9.1% in 2024 and 6.8% in 2025; monthly average 0.66%, spikes in Mar/23 (1.47%) and Jan/24 (1.44%). By department Finance (8.6%) and Operations (8.3%) sit above TI (6.6%) and HR (7.0%), but gaps are small.

**4. One in ten active employees is a pay outlier / Um em cada dez ativos está fora da curva.**
PT: 124 dos 1.140 ativos (10,9%) ganham 15% ou mais acima da média do próprio cargo, um excedente de cerca de R$ 115 mil por mês (2,4% da folha ativa). A taxa é maior em Assistente Fin. (14,3%) e Analista Fin. (13,5%) e menor em Analista RH (4,8%). O maior desvio é o funcionário 1404 (Vendedor, +43,7%).
EN: 124 of 1,140 active employees (10.9%) earn 15%+ above their job average, about BRL 115k per month of excess (2.4% of active payroll). Rates are highest for Assistente Fin. (14.3%) and Analista Fin. (13.5%), lowest for Analista RH (4.8%).

**5. Top earners are mostly outliers too / Os maiores salários são quase todos outliers.**
PT: 13 dos 15 maiores salários por departamento também estão 15%+ acima da média do cargo (inclusive 157 em RH, +40,4%), o que sugere revisar as faixas salariais dentro de cada cargo antes de mexer em cargos inteiros.
EN: 13 of the 15 top-3 salaries per department are also 15%+ above their job average, so review pay bands within each job before touching whole job families.

**Limits / Limites:** synthetic data, uniform hiring dates and no salary raises, so these numbers illustrate the method and do not describe a real company. / dados sintéticos, admissões uniformes e sem reajustes: os números ilustram o método, não uma empresa real.